# Text classification using RNNs

## Pre-processing

In [2]:
import pandas as pd

data = pd.read_csv('../data/petitions.csv')
data.head()

,id,public_petition_text,reason_category
0,3168490,снег на дороге,Благоустройство
1,3219678,очистить кабельный киоск от рекламы,Благоустройство
2,2963920,"Просим убрать все деревья и кустарники, которы...",Благоустройство
3,3374910,Неудовлетворительное состояние парадной - надп...,Содержание МКД
4,3336285,Граффити,Благоустройство


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 59889 entries, 0 to 59888
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id                    59889 non-null  int64
 1   public_petition_text  59889 non-null  str  
 2   reason_category       59889 non-null  str  
dtypes: int64(1), str(2)
memory usage: 1.4 MB


In [7]:
x = data["public_petition_text"]
y = data["reason_category"]

In [ ]:
import re
import nltk
import pymorphy3
from razdel import tokenize
from nltk.corpus import stopwords

nltk.download('stopwords')
russian_stopwords = set(stopwords.words('russian'))

morph = pymorphy3.MorphAnalyzer()

def preprocess_text(text):

    text = text.lower()

    # Удаление URL
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Удаление email
    text = re.sub(r'\S+@\S+', '', text)

    # Замена числительных на специальный токен
    text = re.sub(r'\d+', 'NUM', text)

    # Токенизация через razdel
    tokens = [token.text for token in tokenize(text)]

    # Убирает точки, запятые, скобки, тире, кавычки и смайлики
    clean_tokens = [
        t for t in tokens 
        if re.search(r'[а-яёa-z]', t) or t == 'NUM'
    ]

    # Лемматизация
    lemmatized_words = [morph.parse(word)[0].normal_form for word in clean_tokens]

    # Удаление стоп-слов
    filtered_tokens = [word for word in lemmatized_words if word not in russian_stopwords]

    return filtered_tokens

### Dataset tokenization

In [ ]:
from tqdm.auto import tqdm

tqdm.pandas()

processed_tokens = x.progress_apply(preprocess_text)

processed_tokens = processed_tokens[processed_tokens.apply(len) > 0].reset_index(drop=True)

### Target encoding

In [9]:
y.value_counts()

reason_category
Благоустройство                                                                     34769
Содержание МКД                                                                      14461
Нарушение правил пользования общим имуществом                                        2170
Незаконная информационная и (или) рекламная конструкция                              1831
Фасад                                                                                1483
Повреждения или неисправность элементов уличной инфраструктуры                       1164
Кровля                                                                                825
Водоснабжение                                                                         809
Состояние рекламных или информационных конструкций                                    673
Санитарное состояние                                                                  434
Центральное отопление                                                               

In [13]:
dict_y = {}
for i, val in enumerate(y.value_counts().index, start=1):
    dict_y[val] = i

dict_y

{'Благоустройство': 1,
 'Содержание МКД': 2,
 'Нарушение правил пользования общим имуществом': 3,
 'Незаконная информационная и (или) рекламная конструкция': 4,
 'Фасад': 5,
 'Повреждения или неисправность элементов уличной инфраструктуры': 6,
 'Кровля': 7,
 'Водоснабжение': 8,
 'Состояние рекламных или информационных конструкций': 9,
 'Санитарное состояние': 10,
 'Центральное отопление': 11,
 'Подвалы': 12,
 'Незаконная реализация товаров с торгового оборудования (прилавок, ящик, с земли)': 13,
 'Нарушение порядка пользования общим имуществом': 14,
 'Водоотведение': 15}

In [14]:
y_encoded = y.map(dict_y)

### Separating data

In [15]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.15, random_state=81, shuffle=True)

### RNN (Recurrent Neural Networks)

### LSTM (Long-Short Term Memory)

### GRU (Gated Recurrent Unit)

### Сomparison of metrics

### Custom GRU-block

#### Metrics of custom GRU-block